# Python for Biochemistry
## A Hands-on Workshop — Retrieving Information from the PDB using the Web API

**Author:** Omar Hamouda 

 **Environment:** Python 3.x | Biopython 1.88 | JupyterLab 4.x | Anaconda

---

## 📌 Overview

This notebook covers **programmatic access** to the Protein Data Bank (PDB) through **REST APIs**. Using Python's `requests` library, we will retrieve metadata, search by keyword, and perform **batch structural analysis** using Biopython.

A **REST API** allows different software systems to communicate over the internet by varying parameters in a URL.

---

## 📚 Table of Contents

1. Setup & Imports
2. File Download with `PDBList`
3. PDB Data API — Metadata Retrieval
4. JSON Parsing
5. PDB Search API — Keyword Search
6. Advanced Search Queries
7. Batch Analysis with Biopython

---

## 📁 Data Files

- No pre-existing files required.
- Structures will be **downloaded programmatically** from the RCSB PDB.

## ✏️ Test: Is the Notebook Working?

Quick sanity check: Verify that the `requests` library is available ✅

In [1]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())

Working directory set to: /home/omar/biochemist-python_ORGANIZED


In [ ]:
# Verify requests is installed
import requests
print("requests version:", requests.__version__)

---

## 1️⃣ Setup & Imports

### 🎯 Why?

Before we start hitting APIs, we need to import the tools we'll use:

- **`requests`** — sends HTTP requests to web URLs.
- **`json`** — converts Python dictionaries to JSON (and back).
- **`PDBList`** — Biopython tool for downloading structure files.
- **`MMCIFParser`** — parses mmCIF files into Structure objects.
- **`NeighborSearch`** — finds atoms within a cutoff distance.
- **`Counter`** — counts occurrences (for batch analysis).

> **Golden Rule:** Always import all required libraries **at the top** of the notebook.

In [ ]:
# ─── Standard Library ───
import os
import json
from collections import Counter

# ─── Third-party ───
import requests

# ─── Biopython ───
from Bio.PDB import PDBList
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB import NeighborSearch

---

## 2️⃣ File Download with `PDBList`

### 🎯 Why?

Before analyzing any structure, we need to **download the mmCIF file** from the RCSB PDB.

Instead of manually going to the website and clicking "Download", Biopython's `PDBList` lets us do this **programmatically** — a single function call.

### 🛠️ Function signature

```python
pdb_list.retrieve_pdb_file(
    pdb_id,                    # e.g., "4hhb"
    pdir="data/PDB_files",     # save folder
    file_format="mmCif"        # format
)

Reproducible: Anyone running the notebook downloads the same file.

Batchable: Can loop over dozens of PDB IDs.

Automatic: No manual clicking required.

Note: The PDB ID must be lowercase ("4hhb", not "4HHB")

In [ ]:
# Create an instance of the PDBList class
pdb_list = PDBList()

# Inspect the type
print("Type:", type(pdb_list))

In [ ]:
# PDB ID to download (lowercase!)
pdb_id = "4hhb"

# Download the mmCIF file
pdb_filename = pdb_list.retrieve_pdb_file(
    pdb_id,
    pdir="data/PDB_files",
    file_format="mmCif"
)

print("Downloaded to:", pdb_filename)

In [ ]:
# Verify the file exists
if os.path.exists(pdb_filename):
    print(f"✅ File exists: {pdb_filename}")
    print(f"File size: {os.path.getsize(pdb_filename) / 1024:.2f} KB")
else:
    print(f"❌ File NOT found: {pdb_filename}")

### 📝 Section Summary

In this section, we:
- **Created** a `PDBList` instance from Biopython.
- **Downloaded** the mmCIF file for `4hhb` (Human Deoxyhemoglobin).
- **Verified** the file exists and checked its size.

> **Key takeaway:** `PDBList.retrieve_pdb_file()` makes file downloads **reproducible** and **automatable** — no manual clicking required.

## 3️⃣ PDB Data API

In [ ]:
# URL for the entry 4hhb
url = "https://data.rcsb.org/rest/v1/core/entry/4hhb"

# Send the request
data = requests.get(url)

# Check the status
print("Status code:", data.status_code)
print("Data type:  ", type(data))

In [ ]:
# Convert the response to a Python dictionary
info_4hhb = data.json()

# Inspect the type
print("Type:", type(info_4hhb))

# How many keys?
print("Number of keys:", len(info_4hhb))

In [ ]:
# Show all available keys
for key in info_4hhb.keys():
    print("→", key)

In [ ]:
# Extract specific pieces of information
print("Experiment method:", info_4hhb["exptl"])
print()
print("Keywords:         ", info_4hhb["struct_keywords"])
print()
print("Title:            ", info_4hhb["struct"])

In [ ]:
# Extract detailed entry info
entry_info = info_4hhb["rcsb_entry_info"]

print("Molecular weight:      ", entry_info["molecular_weight"], "kDa")
print("Deposited atom count:  ", entry_info["deposited_atom_count"])
print("Polymer entity count:  ", entry_info["polymer_entity_count"])
print("Nonpolymer components: ", entry_info["nonpolymer_bound_components"])
print("Resolution:            ", entry_info["resolution_combined"][0], "Å")

### 📝 Section Summary

In this section, we:
- **Called** the PDB Data API via `requests.get()`.
- **Checked** the status code (`200` = success).
- **Converted** the JSON response to a Python dictionary via `.json()`.
- **Extracted** specific metadata: title, keywords, method, resolution.

> **Key takeaway:** REST APIs use **URL patterns** to query data. The `requests` library makes this **programmatic** — no web browser needed.

## 4️⃣ Additional Data API Endpoints


In [ ]:
# Query the PubMed endpoint
pubmed_url = "https://data.rcsb.org/rest/v1/core/pubmed/4hhb"
pubmed_data = requests.get(pubmed_url)

print("Status code:", pubmed_data.status_code)

# Convert to dictionary
pubmed_info = pubmed_data.json()
print("Number of keys:", len(pubmed_info))

# Show the keys
for key in pubmed_info.keys():
    print("→", key)

In [ ]:
# Show the ACTUAL keys from the pubmed endpoint
print("Type:", type(pubmed_info))
print("\nAll keys:")
for key in pubmed_info.keys():
    print("  →", key)

In [ ]:
# Extract publication info using the ACTUAL keys we discovered
print("PDB ID:  ", pubmed_info["rcsb_id"])
print("DOI:     ", pubmed_info["rcsb_pubmed_doi"])
print()
print("Abstract (first 300 chars):")
abstract = pubmed_info["rcsb_pubmed_abstract_text"]
print(abstract[:300], "...")

In [ ]:
# Show MeSH descriptors (medical subject headings)
mesh = pubmed_info["rcsb_pubmed_mesh_descriptors"]
print(f"Number of MeSH terms: {len(mesh)}\n")

for term in mesh[:10]:    # first 10 terms
    print("  →", term)

In [ ]:
# Query the interface endpoint for 4hhb
interface_url = "https://data.rcsb.org/rest/v1/core/interface/4hhb/1/1"
interface_data = requests.get(interface_url)

print("Status code:", interface_data.status_code)

# Convert to dictionary
interface_info = interface_data.json()

# Explore the keys FIRST (API may have changed)
print("\nAvailable keys in interface_info:")
for key in interface_info.keys():
    print("  →", key)

In [ ]:
# Extract interface details — explore first
interface_details = interface_info["rcsb_interface_info"]

print("All keys and values in rcsb_interface_info:")
for key, value in interface_details.items():
    print(f"  → {key}: {value}")

In [ ]:
# Display interface details in a clean format
print("═" * 50)
print("  INTERFACE ANALYSIS — 4HHB (Assembly 1, Interface 1)")
print("═" * 50)
print(f"Polymer composition:    {interface_details['polymer_composition']}")
print(f"Interface character:    {interface_details['interface_character']}")
print(f"Interface area:         {round(interface_details['interface_area'], 2)} Å²")
print(f"Interface residues:     {interface_details['num_interface_residues']}")
print(f"Core interface residues:{interface_details['num_core_interface_residues']}")
print("═" * 50)

In [ ]:
# Try different interface IDs — see which ones exist
print("Exploring all interfaces in assembly 1:\n")

for interface_id in [1, 2, 3, 4]:
    url = f"https://data.rcsb.org/rest/v1/core/interface/4hhb/1/{interface_id}"
    response = requests.get(url)
    
    if response.status_code == 200:
        info = response.json()["rcsb_interface_info"]
        area = info["interface_area"]
        char = info["interface_character"]
        print(f"  Interface 1/{interface_id}: ✅ {area:.2f} Å²  ({char})")
    else:
        print(f"  Interface 1/{interface_id}: ❌ not found (status {response.status_code})")

### 📝 Section Summary

In this section, we:
- **Queried** the `pubmed` endpoint and retrieved publication info (DOI, abstract, MeSH terms).
- **Explored** the `interface` endpoint and analyzed protein-protein interfaces.
- **Discovered** 3 interfaces in assembly 1 (α-β contacts).
- **Learned** how to handle API changes — always inspect `keys()` first.

> **Key takeaway:** The PDB Data API offers **multiple endpoints** with the same URL pattern. Changing the endpoint name returns different data about the same PDB entry.

## 5️⃣ PDB Search API


In [ ]:
# Build a search query as a Python dictionary
my_query = {
    "query": {
        "type": "terminal",
        "service": "full_text",
        "parameters": {
            "value": '"oxygen storage"'
        }
    },
    "return_type": "entry"
}

# Inspect the dictionary
print("Query type:", type(my_query))
print("Query content:")
print(my_query)

In [ ]:
# Convert dict to JSON string (required by Search API)
my_query_json = json.dumps(my_query)

print("Type:", type(my_query_json))
print()
print("JSON string:")
print(my_query_json)

In [ ]:
# Build the full URL and send the request
url = f"https://search.rcsb.org/rcsbsearch/v2/query?json={my_query_json}"

print("Sending request...")
response = requests.get(url)
print("Status code:", response.status_code)

In [ ]:
# Convert to dictionary
results = response.json()

# Show overview
print("Result type:  ", results["result_type"])
print("Total count:  ", results["total_count"])
print("Returned:     ", len(results["result_set"]))
print()
print("First result:")
print("  ", results["result_set"][0])

In [ ]:
# Show all 10 results
print("Search results (first 10):\n")
for i, result in enumerate(results["result_set"], start=1):
    pdb_id = result["identifier"]
    score  = result["score"]
    print(f"  {i:2d}. {pdb_id}   (score: {score:.4f})")

In [ ]:
# Take the first result and get its details via Data API
first_pdb_id = results["result_set"][0]["identifier"]
print("First result PDB ID:", first_pdb_id)

# Query the Data API for this entry
data_url = f"https://data.rcsb.org/rest/v1/core/entry/{first_pdb_id}"
detail_response = requests.get(data_url)
detail_info = detail_response.json()

print("\nTitle:", detail_info["struct"]["title"])

### 📝 Section Summary

In this section, we:
- **Built** a search query as a Python dictionary.
- **Converted** it to JSON with `json.dumps()`.
- **Sent** the query to the PDB Search API via `requests.get()`.
- **Parsed** the JSON response and extracted PDB IDs.
- **Combined** Search API + Data API to get full entry details.

> **Key takeaway:** The Search API lets you **discover** PDB entries matching a keyword; the Data API lets you **describe** them.

---

## 6️⃣ Advanced Search Queries



In [ ]:
# Build an advanced query with sorting and pagination
advanced_query = {
    "query": {
        "type": "terminal",
        "service": "text",
        "parameters": {
            "attribute": "struct_keywords.pdbx_keywords",
            "operator": "contains_phrase",
            "value": '"oxygen storage"'
        }
    },
    "request_options": {
        "paginate": {
            "start": 0,
            "rows": 50
        },
        "sort": [
            {
                "sort_by": "rcsb_accession_info.initial_release_date",
                "direction": "asc"
            }
        ]
    },
    "return_type": "entry"
}

print("Query dict created.")
print("Rows requested:", advanced_query["request_options"]["paginate"]["rows"])

In [ ]:
# Convert to JSON and send
advanced_query_json = json.dumps(advanced_query)

url = f"https://search.rcsb.org/rcsbsearch/v2/query?json={advanced_query_json}"
response = requests.get(url)

print("Status code:", response.status_code)

# Convert to dict
results_v2 = response.json()
print("Total count:  ", results_v2["total_count"])
print("Returned now: ", len(results_v2["result_set"]))

In [ ]:
# Show first 10 results (sorted by date, oldest first)
print("First 10 results (sorted by release date, oldest first):\n")
for i, result in enumerate(results_v2["result_set"][:10], start=1):
    pdb_id = result["identifier"]
    print(f"  {i:2d}. {pdb_id}")

In [ ]:
# Try descending sort (newest first)
query_desc = {
    "query": advanced_query["query"],
    "request_options": {
        "paginate": {"start": 0, "rows": 10},
        "sort": [
            {
                "sort_by": "rcsb_accession_info.initial_release_date",
                "direction": "desc"
            }
        ]
    },
    "return_type": "entry"
}

query_desc_json = json.dumps(query_desc)
url = f"https://search.rcsb.org/rcsbsearch/v2/query?json={query_desc_json}"
results_desc = requests.get(url).json()

print("Newest 10 results:\n")
for i, result in enumerate(results_desc["result_set"], start=1):
    print(f"  {i:2d}. {result['identifier']}")

### 📝 Section Summary

In this section, we:
- **Built** an advanced query using the `text` service (attribute-specific search).
- **Controlled** result count via `paginate.rows` (from 10 → 50).
- **Sorted** results by release date (`asc` for oldest, `desc` for newest).
- **Compared** `full_text` vs `text` searches.

> **Key takeaway:** `request_options` gives full control over **pagination** and **sorting** — essential for batch analysis.

---

## 7️⃣ Batch Analysis with Biopython



In [ ]:
# Create a directory for batch files
batch_dir = "pdb_files"
os.makedirs(batch_dir, exist_ok=True)
print(f"Directory '{batch_dir}' is ready.")

# Download all files
from Bio.PDB import PDBList

pdb_list = PDBList()

# Counter for progress
downloaded = 0
skipped = 0

for result in results_v2["result_set"]:
    pdb_id = result["identifier"].lower()
    file_path = os.path.join(batch_dir, f"{pdb_id}.cif")
    
    if os.path.exists(file_path):
        skipped += 1
        continue
    
    pdb_list.retrieve_pdb_file(pdb_id, pdir=batch_dir, file_format="mmCif")
    downloaded += 1

print(f"\n✅ Downloaded: {downloaded}")
print(f"⏭️  Skipped (already exists): {skipped}")
print(f"📁 Total files in '{batch_dir}': {len(os.listdir(batch_dir))}")

In [ ]:
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB import NeighborSearch
from collections import Counter

# Setup
parser = MMCIFParser(QUIET=True)
cutoff_distance = 5   # Å

# Dictionary to store neighbors per PDB ID
residue_neighbors = {}

# Loop over all files
for i, result in enumerate(results_v2["result_set"], start=1):
    pdb_id = result["identifier"].lower()
    file_path = os.path.join(batch_dir, f"{pdb_id}.cif")
    
    if not os.path.exists(file_path):
        print(f"  [{i:2d}/50] {pdb_id}: ❌ file missing")
        continue
    
    try:
        # Parse structure
        structure = parser.get_structure(pdb_id, file_path)
        atoms = list(structure.get_atoms())
        
        # Create NeighborSearch
        ns = NeighborSearch(atoms)
        
        # Find all iron atoms and their neighbors
        neighbor_list = []
        for atom in atoms:
            if atom.element == "FE":
                iron_residue = atom.get_parent()
                neighbors = ns.search(atom.get_coord(), cutoff_distance)
                
                for neighbor in neighbors:
                    residue = neighbor.get_parent()
                    if residue != iron_residue:
                        neighbor_list.append(residue)
        
        # Store unique residues
        residue_neighbors[pdb_id] = set(neighbor_list)
        print(f"  [{i:2d}/50] {pdb_id}: ✅ {len(set(neighbor_list))} unique neighbors")
    
    except Exception as e:
        print(f"  [{i:2d}/50] {pdb_id}: ❌ error — {e}")

print(f"\n✅ Processed: {len(residue_neighbors)} structures")

In [ ]:
# Count the most common residue combinations
combination_counts = Counter()

for pdb_id, neighbors in residue_neighbors.items():
    # Get residue names only
    resname = [x.get_resname() for x in neighbors if x.get_resname()]
    
    # Count occurrences per residue
    res_counts = Counter(resname)
    
    # Standardize: sorted tuple of (resname, count)
    combination = tuple(sorted(res_counts.items()))
    
    # Update global counter
    combination_counts.update([combination])

# Show top 5
print("Top 5 most common residue combinations around iron:\n")
for combo, count in combination_counts.most_common(5):
    combo_str = ', '.join([f"{c} {r}" for r, c in combo])
    print(f"  {combo_str}   →   appeared in {count} structures")

### 📝 Section Summary

In this section, we:
- **Downloaded** 50 mmCIF files programmatically.
- **Parsed** each structure and searched for iron neighbors.
- **Collected** unique residues per structure.
- **Counted** the most common residue combinations using `Counter`.
- **Identified** `2 HIS` as the dominant iron-coordinating motif.

> **Key takeaway:** Batch analysis combines **Search API** + **File Download** + **Biopython** + **Python data structures** to extract **biological insights** from hundreds of structures.

---

## 📝 Summary & Key Takeaways

In this notebook, we applied **REST API principles** to access the Protein Data Bank programmatically. We combined **file downloads**, **metadata retrieval**, **keyword search**, and **batch analysis** to answer a real biological question.

### What we covered

- **REST API** — understood how URLs encode queries.
- **PDBList** — downloaded mmCIF files programmatically.
- **Data API** — retrieved entry metadata as JSON.
- **Additional Endpoints** — explored `pubmed` and `interface` endpoints.
- **Search API** — performed keyword searches with `full_text` and `text`.
- **Advanced Queries** — controlled pagination (`rows`) and sorting (`asc`/`desc`).
- **Batch Analysis** — processed 50 structures with Biopython.
- **Counter** — extracted the most common residue combinations.

### Result

We identified the **most common iron-coordinating motifs** across 50 oxygen-storage proteins. The dominant pattern — **`2 HIS + 1 HOH + 1 VAL`** — confirms that **histidine is the primary iron-coordinating residue** in heme proteins, consistent with what we found in myoglobin .

---

## 🚀 Next Steps

The workflow we built here opens up several directions:

- **Larger datasets** — apply the same pipeline to thousands of PDB entries.
- **Other metals** — replace `"FE"` with `"ZN"`, `"MG"`, `"CA"`, `"MN"` for other metalloproteins.
- **Other properties** — analyze ligand types, binding pockets, or secondary structure motifs.
- **Visualization** — plot the distribution of distances, residue types, or interface areas with `matplotlib`.
- **Machine learning** — use the extracted features for classification tasks (e.g., predict metal-binding sites).
- **Combine with Lesson 10** — verify batch results with detailed single-structure analysis.

### 📌 Useful Resources

| Resource | Link |
|----------|------|
| **RCSB PDB REST API** | https://data.rcsb.org/redoc/ |
| **Search API Documentation** | https://search.rcsb.org/ |
| **Biopython Tutorial** | https://biopython.org/wiki/Documentation |
| **PDB101** | https://pdb101.rcsb.org/ |